# Trabalho de Séries Temporais — Grupo 5 (PLS)

O notebook tem 4 partes:

1. **Bases** — lê cada base e registra na lista `bases` (é aqui que se adiciona base nova)
2. **Análise prévia** — qualidade, STL, estacionariedade, ACF/PACF e validação das
   exógenas. Não é só diagnóstico: é aqui que `m`, `d`, `D` e o conjunto final de
   exógenas são **decididos** e gravados na base
3. **Pipeline** — features, otimização e walk-forward
4. **Comparação** — MAE, ranking, vitórias, resíduos e importância das features

A parte 3 não compara nada: ela só produz as previsões fora da amostra de
5 bases × 4 modelos. A comparação é a parte 4.

Duas regras valem no notebook inteiro: `random_state=42` em tudo que sorteia,
e separação treino/teste **70-30**.

Toda a parte 2 roda **apenas sobre treino + validação**. O conjunto de teste não
participa de nenhuma decisão — nem da escolha de `m`, nem de `d`/`D`, nem de
quais exógenas entram.

In [ ]:
import os
# threads do BLAS em 1: quem paraleliza aqui e o joblib, nao a algebra linear
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from scipy.stats import pearsonr

from sklearn.ensemble import RandomForestRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42     # regra do grupo: vale para tudo que sorteia
P_TESTE = 0.30        # regra do grupo: separacao treino/teste 70-30

# o teste final usa TODAS as origens. a otimizacao usa no maximo esta
# quantidade, espacadas por toda a janela de validacao -- senao a grade
# de hiperparametros multiplica por centenas de ajustes por combinacao.
MAX_ORIGENS_VALIDACAO = 30

# maior periodo sazonal que o SARIMAX aguenta: m grande explode o espaco de
# estados. acima disso, o ciclo fica por conta das features de calendario.
M_MAX = 30

# abaixo desta forca a serie e tratada como nao sazonal (m = 1)
FORCA_SAZONAL_MIN = 0.05

np.random.seed(RANDOM_STATE)

# 1. Bases

Cada base entra como um dicionário. O `df` precisa chegar aqui **já tratado**:
índice de data, frequência regular, sem furo no alvo.

As externas vão em duas listas, e isso muda como o pipeline as usa:

- `exog_conhecidas` — o valor futuro já é sabido na data da previsão
  (feriado, calendário, promoção planejada). Entra com o valor da própria data.
- `exog_defasadas` — não se sabe o valor futuro (temperatura observada,
  indicador divulgado com atraso). Entra só defasada em `h`.

O 70-30 é aplicado duas vezes: 30% final da série é o **teste**, e 30% final do
que sobrou é a **validação**, onde os hiperparâmetros são escolhidos.

In [ ]:
bases = []

def add_base(nome, df, alvo, h, exog_conhecidas=(), exog_defasadas=(),
             n_validacao=None, n_teste=None):
    '''Registra uma base na lista global `bases`.
    
        `m` fica None de proposito: quem preenche e o `decidir_m`, na parte 2,
        a partir da forca da sazonalidade. Os cortes 70-30 sao calculados aqui.
        '''
    n = len(df)
    n_teste = n_teste or round(P_TESTE * n)                  # 30% final -> teste
    n_treino = n - n_teste
    n_validacao = n_validacao or round(P_TESTE * n_treino)   # 30% do treino -> validacao

    base = {
        "nome": nome,
        "df": df.sort_index(),
        "alvo": alvo,
        "m": None,                                 # preenchido por decidir_m (parte 2)
        "h": h,                                    # horizonte de previsao
        "exog_conhecidas": list(exog_conhecidas),
        "exog_defasadas": list(exog_defasadas),
        "n_validacao": n_validacao,
        "n_teste": n_teste,
    }
    bases.append(base)
    return base

As cinco bases congeladas ficam em `bases/`, geradas pelo `prepare_bases.ipynb`
a partir de `bases/raw/`. Os alvos e as regras de disponibilidade vêm de lá.

**Para adicionar ou trocar uma base: copia uma célula abaixo, troca a leitura e
o `add_base`.** O resto do notebook roda sozinho em cima da lista `bases`.

In [ ]:
def ler(arquivo, freq=None):
    '''Le uma base preparada. `freq=None` para acoes: o indice sao sessoes
        de pregao observadas, e aplicar asfreq inventaria precos de feriado.'''
    df = pd.read_csv(f"bases/{arquivo}", parse_dates=["Date"], index_col="Date")
    return df.asfreq(freq) if freq else df      # acoes: sessoes observadas, sem asfreq

### 1. Delhi — temperatura média diária

Alvo `meantemp`. As três covariáveis climáticas não são conhecidas com
antecedência, então entram defasadas em `h`.

A sazonalidade real aqui é anual e muito forte, mas `m=365` não é computável no
SARIMAX — a parte 2 vai escolher o melhor `m` dentro do `M_MAX` e registrar o
que ficou de fora. O ciclo anual sobra para as features cíclicas de mês e
semana, que só Random Forest e PLS recebem.

In [ ]:
add_base(
    nome="delhi_temperatura",
    df=ler("daily_delhi_climate_prepared.csv", "D"),
    alvo="meantemp",
    h=7,
    exog_defasadas=["humidity", "wind_speed", "meanpressure"],
)

### 2. Pilgrim's Pride — preço de fechamento

Alvo `Close`. Open, High, Low e Volume são da **mesma sessão**: usar o valor
do próprio dia seria vazamento, então todos entram defasados em `h`.

Índice de sessões de pregão, sem `asfreq` — não se inventa preço de fim de
semana. Os lags são por sessão, não por dia de calendário.

In [ ]:
add_base(
    nome="pilgrims_close",
    df=ler("pilgrims_pride_prepared.csv"),
    alvo="Close",
    h=5,
    exog_defasadas=["Open", "High", "Low", "Volume"],
)

### 3. Microsoft — abertura da sessão

Alvo `target_open`. `close_lag_1` e `volume_lag_1` já vêm defasados de uma
sessão pelo `prepare_bases`, então com `h=1` eles **são conhecidos na origem**:
ao prever a abertura de amanhã, o fechamento e o volume de hoje já aconteceram.

Isso só vale enquanto `h=1`. Com `h>1` eles teriam que virar `exog_defasadas`.

In [ ]:
add_base(
    nome="microsoft_open",
    df=ler("microsoft_stock_prepared.csv"),
    alvo="target_open",
    h=1,
    exog_conhecidas=["close_lag_1", "volume_lag_1"],
)

### 4. Sales — lucro diário

Alvo `Profit`, soma das transações do dia. `order_quantity` e
`transaction_count` são do próprio dia: só se sabe depois que o dia acabou,
então entram defasadas — é o que o `prepare_bases` determina.

In [ ]:
add_base(
    nome="sales_profit",
    df=ler("sales_prepared.csv", "D"),
    alvo="Profit",
    h=7,
    exog_defasadas=["order_quantity", "transaction_count"],
)

### 5. Brasil — vitórias por ano

Alvo `victories`, contagem anual. `m=4` pelo ciclo de Copa do Mundo, que é a
única sazonalidade plausível numa série anual (força 0.23 contra 0.06 em m=2).

`matches_played` e `friendly_matches` são contados depois que o ano termina,
então entram defasados — mesmo critério do `prepare_bases`. Vale notar a
relação mecânica: mais jogos disputados tende a significar mais vitórias.

Série curta: 109 observações. As features de calendário caem fora sozinhas,
porque todas as datas são 1º de janeiro e viram colunas constantes.

In [ ]:
add_base(
    nome="brasil_vitorias",
    df=ler("brazil_prepared.csv", "YS"),
    alvo="victories",
    h=1,
    exog_defasadas=["matches_played", "friendly_matches"],
)

In [ ]:
pd.DataFrame([{
    "base": b["nome"],
    "obs": len(b["df"]),
    "inicio": b["df"].index.min().date(),
    "fim": b["df"].index.max().date(),
    "freq": b["df"].index.freqstr,
    "m": b["m"],
    "h": b["h"],
    "treino": len(b["df"]) - b["n_teste"] - b["n_validacao"],
    "validacao": b["n_validacao"],
    "teste": b["n_teste"],
    "conhecidas": ", ".join(b["exog_conhecidas"]) or "-",
    "defasadas": ", ".join(b["exog_defasadas"]) or "-",
} for b in bases])

# 2. Análise prévia

Antes de modelar: entender cada base e **decidir** com base no que se vê.
Cada seção aqui termina gravando uma decisão no dicionário da base —
`m`, `d`, `D` e quais exógenas sobrevivem.

Tudo roda sobre **treino + validação**. O conjunto de teste não entra em
nenhuma dessas contas: escolher `d` olhando o teste é vazamento igual a
treinar nele.

In [ ]:
def dados_analise(base):
    '''Treino + validacao. Toda a parte 2 usa isto, nunca a serie inteira.'''
    # tudo menos o teste: e so isso que pode influenciar qualquer decisao
    return base["df"].iloc[:-base["n_teste"]]

def serie_analise(base):
    '''So a variavel-alvo da janela de analise, ja em float.'''
    return dados_analise(base)[base["alvo"]].astype(float)

## 2.1 Qualidade dos dados

Ausentes, duplicidades, irregularidade temporal e valores atípicos — os quatro
itens que o enunciado cobra na seção 5.1.

Outliers são **contados, não removidos**: em preço de ação e em clima, o ponto
extremo costuma ser o evento real que mais importa prever. Remover seria
apagar informação. A contagem entra no relatório como caracterização.

In [ ]:
def diagnosticar(base):
    '''Ausentes, duplicidades, irregularidade temporal e outliers de uma base.
    
        Outliers sao contados pelo criterio IQR e NAO removidos: em preco de acao
        e em clima, o ponto extremo costuma ser o evento que mais importa prever.
        '''
    df = dados_analise(base)
    y = df[base["alvo"]].astype(float)

    # irregularidade temporal: quantos intervalos diferentes existem entre
    # observacoes consecutivas (acoes tem 2-3 por causa de fim de semana)
    intervalos = df.index.to_series().diff().dropna().value_counts()

    q1, q3 = y.quantile([0.25, 0.75])
    iqr = q3 - q1
    fora = ((y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)).sum()

    return {
        "base": base["nome"],
        "obs": len(df),
        "inicio": df.index.min().date(),
        "fim": df.index.max().date(),
        "datas_duplicadas": int(df.index.duplicated().sum()),
        "alvo_ausente": int(y.isna().sum()),
        "exog_ausente": int(df[base["exog_conhecidas"] + base["exog_defasadas"]].isna().sum().sum()),
        "intervalos_distintos": len(intervalos),
        "intervalo_comum": str(intervalos.index[0]),
        "outliers_iqr": int(fora),
        "pct_outliers": round(100 * fora / len(y), 1),
        "zeros": int((y == 0).sum()),
        "min": round(y.min(), 2),
        "max": round(y.max(), 2),
    }


pd.DataFrame([diagnosticar(b) for b in bases]).set_index("base").T

## 2.2 A série-alvo e as externas

In [ ]:
def plotar_base(base):
    '''Serie-alvo e cada externa, rotuladas por papel (conhecida ou defasada).'''
    df = dados_analise(base)
    exog = base["exog_conhecidas"] + base["exog_defasadas"]
    colunas = [base["alvo"]] + exog

    fig, axes = plt.subplots(len(colunas), 1, figsize=(13, 2.2 * len(colunas)),
                             sharex=True)
    axes = np.atleast_1d(axes)

    for ax, col in zip(axes, colunas):
        cor = "steelblue" if col == base["alvo"] else "#9aa4ad"
        ax.plot(df.index, df[col].astype(float), color=cor, linewidth=0.9)
        papel = "ALVO" if col == base["alvo"] else (
            "conhecida" if col in base["exog_conhecidas"] else "defasada")
        ax.set_title(f"{col}  ({papel})", loc="left", fontsize=9)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)

    fig.suptitle(base["nome"], y=1.0, fontsize=12)
    plt.tight_layout()
    plt.show()


for b in bases:
    plotar_base(b)

## 2.3 STL e força da sazonalidade → decide `m`

A força da sazonalidade é `1 - Var(resíduo) / Var(sazonal + resíduo)`: perto de
1 a componente sazonal domina, perto de 0 ela é ruído.

`m_max` limita a busca. Não é detalhe metodológico, é computacional: o SARIMAX
com `m=365` tem centenas de estados e não termina. Quando o `m` escolhido não é
o de maior força, isso fica registrado e vira discussão no relatório — o ciclo
mais longo passa a ser responsabilidade das features cíclicas de calendário.

In [ ]:
def forca_sazonalidade(serie, m):
    '''Forca da sazonalidade: 1 - Var(residuo) / Var(sazonal + residuo).
    
        Perto de 1 a componente sazonal domina; perto de 0 (ou negativa) ela e
        indistinguivel de ruido.
        '''
    res = STL(serie, period=m, robust=True).fit()
    return 1 - np.var(res.resid) / np.var(res.seasonal + res.resid)


def ranking_m(serie, candidatos=(2, 3, 4, 5, 6, 7, 12, 14, 21, 30, 52, 90, 252, 365),
              min_ciclos=3):
    '''Forca da sazonalidade para varios periodos candidatos, do maior ao menor.
    
        Candidatos com menos de `min_ciclos` ciclos completos sao descartados:
        sem repeticao suficiente a STL nao consegue separar o ciclo do ruido.
        '''
    linhas = []
    for m in candidatos:
        if len(serie) / m < min_ciclos:
            continue
        try:
            linhas.append({"m": m, "forca": forca_sazonalidade(serie, m)})
        except Exception:
            continue
    return pd.DataFrame(linhas).sort_values("forca", ascending=False).reset_index(drop=True)


def decidir_m(base, m_max=M_MAX, forca_min=FORCA_SAZONAL_MIN):
    '''Escolhe e grava o periodo sazonal da base.
    
        Pega o `m` de maior forca que caiba em `m_max` (limite computacional do
        SARIMAX). Se nem o melhor passar de `forca_min`, grava m=1: a serie e
        tratada como nao sazonal em vez de receber um ciclo inventado.
        '''
    serie = serie_analise(base)
    tabela = ranking_m(serie)

    viaveis = tabela[tabela["m"] <= m_max]
    melhor_m = int(viaveis.iloc[0]["m"])
    melhor_forca = float(viaveis.iloc[0]["forca"])

    # forca baixa demais: nao ha sazonalidade para modelar. m=1 desliga o termo
    # sazonal do SARIMAX e do Holt-Winters, em vez de ajustar um ciclo inventado.
    if melhor_forca < forca_min:
        base["m"], base["m_origem"] = 1, "sem sazonalidade"
        base["forca_sazonal"] = melhor_forca
    else:
        base["m"], base["m_origem"] = melhor_m, "maior forca dentro do M_MAX"
        base["forca_sazonal"] = melhor_forca

    base["m_ideal"] = int(tabela.iloc[0]["m"])
    base["forca_ideal"] = float(tabela.iloc[0]["forca"])
    base["m_ideal_viavel"] = base["m_ideal"] <= m_max

    return tabela

In [ ]:
def stl_da_base(base):
    '''STL em quatro paineis: observado, tendencia, sazonal e residuo.'''
    serie = serie_analise(base)
    # m=1 significa "sem sazonalidade para modelar", mas a STL ainda precisa de
    # um periodo: usa o de maior forca so para o grafico do relatorio
    periodo = base["m"] if base["m"] >= 2 else base["m_ideal"]
    res = STL(serie, period=periodo, robust=True).fit()

    fig, axes = plt.subplots(4, 1, figsize=(13, 8), sharex=True)
    for ax, (dado, titulo) in zip(axes, [
        (serie, "observado"), (res.trend, "tendencia"),
        (res.seasonal, f"sazonal (period={periodo})"), (res.resid, "residuo")
    ]):
        ax.plot(dado.index, dado.values, color="steelblue", linewidth=0.9)
        ax.set_title(titulo, loc="left", fontsize=9)
        for lado in ("top", "right"):
            ax.spines[lado].set_visible(False)

    fig.suptitle(f"STL - {base['nome']}", y=1.0, fontsize=12)
    plt.tight_layout()
    plt.show()
    return res


def fases_da_tendencia(base, limiar=0.15, min_pontos=None):
    '''Quebra a tendencia da STL em fases de alta, queda e estabilidade.

    O enunciado pede a interpretacao do componente de tendencia indicando
    crescimento, queda, estabilidade e mudancas de direcao. Ler isso no olho
    e subjetivo; aqui a inclinacao e medida e classificada.

    A inclinacao de cada ponto e normalizada pelo desvio padrao da propria
    tendencia, entao `limiar` fica em unidades comparaveis entre bases:
    abaixo dele o movimento e pequeno demais para ser chamado de direcao.
    Fases mais curtas que `min_pontos` sao absorvidas pela fase anterior,
    para nao picotar a leitura em dezenas de trechos.
    '''
    serie = serie_analise(base)
    periodo = base["m"] if base["m"] >= 2 else base["m_ideal"]
    tendencia = STL(serie, period=periodo, robust=True).fit().trend

    min_pontos = min_pontos or max(3, len(tendencia) // 25)

    # inclinacao suavizada, em desvios-padrao da tendencia por passo
    inclinacao = tendencia.diff().rolling(min_pontos, center=True).mean()
    escala = tendencia.diff().std()
    direcao = pd.Series("estavel", index=tendencia.index)
    direcao[inclinacao > limiar * escala] = "alta"
    direcao[inclinacao < -limiar * escala] = "queda"

    # agrupa pontos consecutivos de mesma direcao
    blocos = (direcao != direcao.shift()).cumsum()

    fases = []
    for _, grupo in direcao.groupby(blocos):
        if len(grupo) < min_pontos:
            continue
        inicio, fim = grupo.index[0], grupo.index[-1]
        fases.append({
            "base": base["nome"],
            "fase": grupo.iloc[0],
            "inicio": inicio.date(),
            "fim": fim.date(),
            "pontos": len(grupo),
            "variacao": round(float(tendencia[fim] - tendencia[inicio]), 2),
            "variacao_pct": round(100 * float(tendencia[fim] / tendencia[inicio] - 1), 1)
                            if tendencia[inicio] != 0 else np.nan,
        })

    return pd.DataFrame(fases)


for b in bases:
    tabela = decidir_m(b)
    print(f"[{b['nome']}] m = {b['m']} ({b['m_origem']}), forca {b['forca_sazonal']:.3f}")

    if not b["m_ideal_viavel"]:
        print(f"   fora do M_MAX={M_MAX}: m={b['m_ideal']} teria forca "
              f"{b['forca_ideal']:.3f} - fica por conta das features de calendario")

    print(tabela.head(5).to_string(index=False))
    stl_da_base(b)

### Fases da tendência

Insumo para a interpretação escrita que o enunciado pede: onde a tendência
sobe, cai, fica estável e onde muda de direção — medido, não estimado no olho.

In [ ]:
fases = pd.concat([fases_da_tendencia(b) for b in bases], ignore_index=True)
fases.to_csv("resultados/fases_tendencia.csv", index=False)
fases

## 2.4 Estacionariedade e ACF/PACF → decide `d` e `D`

ADF e KPSS têm hipóteses nulas opostas, então só concordam quando o caso é
claro: exigir os dois é mais conservador que confiar em um. `d` é o número de
diferenças até os dois concordarem; `D` testa se uma diferença sazonal adicional
ainda ajuda.

O que sai daqui entra direto na grade do SARIMAX, em vez de ficar chutado.

In [ ]:
def eh_estacionaria(serie):
    '''Estacionaria so quando ADF e KPSS concordam.
    
        As hipoteses nulas sao opostas (ADF testa raiz unitaria, KPSS testa
        estacionariedade), entao exigir os dois e mais conservador que confiar
        em um so.
        '''
    # ADF: p < 0.05 -> estacionaria | KPSS: p > 0.05 -> estacionaria
    return adfuller(serie)[1] < 0.05 and kpss(serie)[1] > 0.05

def sugerir_d(serie, max_d=2):
    '''Quantas diferencas simples ate ADF e KPSS concordarem.'''
    s = serie.dropna()
    for d in range(max_d + 1):
        if eh_estacionaria(s):
            return d
        s = s.diff().dropna()
    return max_d

def sugerir_D(serie, m, d):
    '''Decide se cabe uma diferenca sazonal, depois das `d` simples.

    So retorna 1 quando a serie AINDA nao esta estacionaria apos `d`. Se `d` ja
    resolveu, diferenciar de novo sobre-diferencia: infla a variancia do erro e
    piora a previsao.
    '''
    if m < 2:
        return 0

    s = serie.diff(d).dropna() if d > 0 else serie

    # se `d` ja resolveu, diferenca sazonal so sobre-diferencia: infla a
    # variancia do erro e piora a previsao. so vale se ainda nao estiver ok.
    if eh_estacionaria(s):
        return 0

    sd = s.diff(m).dropna()
    if len(sd) < 2 * m:
        return 0
    return 1 if eh_estacionaria(sd) else 0


def decidir_dD(base):
    '''Grava `d` e `D` na base e devolve os p-valores para a tabela.'''
    serie = serie_analise(base)
    base["d"] = sugerir_d(serie)
    base["D"] = sugerir_D(serie, base["m"], base["d"])

    adf_p = adfuller(serie)[1]
    kpss_p = kpss(serie)[1]
    return {"base": base["nome"], "ADF_p": round(adf_p, 4), "KPSS_p": round(kpss_p, 4),
            "estacionaria": adf_p < 0.05 and kpss_p > 0.05, "d": base["d"], "D": base["D"]}

In [ ]:
def acf_pacf(base):
    '''ACF e PACF da serie ja diferenciada em `d`.
    
        Sao elas que sugerem as ordens q (corte na ACF) e p (corte na PACF) que
        a grade do SARIMAX vai varrer.
        '''
    serie = serie_analise(base)
    dif = serie.diff(base["d"]).dropna() if base["d"] > 0 else serie
    lags = max(1, min(40, len(dif) // 2 - 1))

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
    plot_acf(dif, lags=lags, ax=axes[0])
    plot_pacf(dif, lags=lags, ax=axes[1], method="ywm")
    axes[0].set_title(f"ACF (d={base['d']})", loc="left")
    axes[1].set_title(f"PACF (d={base['d']})", loc="left")
    fig.suptitle(base["nome"], y=1.04, fontsize=12)
    plt.tight_layout()
    plt.show()


estacionariedade = pd.DataFrame([decidir_dD(b) for b in bases]).set_index("base")
print(estacionariedade.to_string())

for b in bases:
    acf_pacf(b)

## 2.5 Validação das exógenas → decide quais entram

Duas perguntas, nesta ordem:

**A externa explica o alvo?** Correlação de Pearson com `r` e p-valor. A
correlação é medida na versão **já defasada**, que é como a variável realmente
entra no modelo — correlacionar o valor do mesmo dia mediria uma relação que o
modelo nunca vai poder usar.

**As externas são redundantes entre si?** VIF sobre as externas padronizadas.
VIF alto significa que uma é combinação linear das outras: os coeficientes do
SARIMAX ficam instáveis e a interpretação vira ruído. Poda iterativa, removendo
a de maior VIF até todas ficarem abaixo de 10.

In [ ]:
def colunas_exog(base):
    '''Mapa (coluna na matriz, coluna na base, papel) de cada externa.

    Existe para que ninguem deduza o nome original cortando o sufixo `_lag_h`
    por string: uma base pode ter uma coluna que JA se chama `close_lag_1`, e
    com h=1 esse corte comeria parte do nome de verdade.
    '''
    return ([(c, c, 'conhecida') for c in base['exog_conhecidas']] +
            [(f"{c}_lag_{base['h']}", c, 'defasada') for c in base['exog_defasadas']])


def matriz_exog(base, df):
    '''Externas no formato do statsmodels, respeitando a disponibilidade.
    
        Conhecidas entram com o valor da propria data; defasadas entram deslocadas
        em `h`. O bfill so preenche as primeiras `h` linhas abertas pelo shift.
        '''
    colunas = {}
    for na_matriz, na_base, papel in colunas_exog(base):
        serie = df[na_base].astype(float)
        colunas[na_matriz] = serie if papel == 'conhecida' else serie.shift(base['h'])

    return pd.DataFrame(colunas, index=df.index).bfill() if colunas else None


def correlacoes(base):
    '''Pearson de cada externa contra o alvo, na versao ja defasada.
    
        Correlacionar o valor do mesmo dia mediria uma relacao que o modelo nunca
        vai poder usar na previsao.
        '''
    df = dados_analise(base)
    y = df[base["alvo"]].astype(float)
    exog = matriz_exog(base, df)
    if exog is None:
        return pd.DataFrame()

    linhas = []
    for na_matriz, na_base, papel in colunas_exog(base):
        r, p = pearsonr(exog[na_matriz], y)
        linhas.append({
            'base': base['nome'],
            'exogena': na_matriz,
            'papel': papel,
            "r": round(r, 4),
            "p_valor": round(p, 6),
            "significativa": p < 0.05,
        })
    return pd.DataFrame(linhas)


def vif(exog):
    '''Variance Inflation Factor das externas padronizadas.
    
        VIF alto = a variavel e quase combinacao linear das outras, o que deixa os
        coeficientes instaveis e a interpretacao sem sentido.
        '''
    X = pd.DataFrame(StandardScaler().fit_transform(exog),
                     columns=exog.columns, index=exog.index)
    X = add_constant(X)
    return pd.Series(
        [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
        index=X.columns,
    ).drop("const")

In [ ]:
def decidir_exogenas(base, limite_vif=10.0):
    '''Poda por colinearidade e grava quais externas sobrevivem.
    
        Remove iterativamente a de maior VIF ate todas ficarem sob o limite,
        recalculando a cada passo, e atualiza as listas de externas da base.
        '''
    df = dados_analise(base)
    exog = matriz_exog(base, df)
    if exog is None or exog.shape[1] < 2:
        base["exog_removidas"] = []
        return pd.DataFrame()

    mantidas = list(exog.columns)
    removidas = []

    # poda iterativa: tira a pior, recalcula, repete
    while len(mantidas) > 1:
        v = vif(exog[mantidas])
        if v.max() <= limite_vif:
            break
        pior = v.idxmax()
        removidas.append({"exogena": pior, "vif": round(v.max(), 1)})
        mantidas.remove(pior)

    v_final = vif(exog[mantidas]) if len(mantidas) > 1 else pd.Series({mantidas[0]: 1.0})

    # traduz de volta pelo mapa, nunca por corte de string
    na_base = {matriz: origem for matriz, origem, _ in colunas_exog(base)}
    vivos = {na_base[c] for c in mantidas}

    base['exog_removidas'] = [na_base[r['exogena']] for r in removidas]
    base['exog_conhecidas'] = [c for c in base['exog_conhecidas'] if c in vivos]
    base['exog_defasadas'] = [c for c in base['exog_defasadas'] if c in vivos]

    return pd.DataFrame([
        {"base": base["nome"], "exogena": c, "VIF": round(v_final[c], 2), "decisao": "mantida"}
        for c in mantidas
    ] + [
        {"base": base["nome"], "exogena": r["exogena"], "VIF": r["vif"], "decisao": "removida"}
        for r in removidas
    ])

In [ ]:
corr = pd.concat([correlacoes(b) for b in bases], ignore_index=True)
print("Correlacao de Pearson com o alvo (exogenas ja defasadas)")
print(corr.to_string(index=False))

In [ ]:
vifs = pd.concat([decidir_exogenas(b) for b in bases], ignore_index=True)
print("VIF e decisao")
print(vifs.to_string(index=False))

for b in bases:
    if b.get("exog_removidas"):
        print(f"[{b['nome']}] removidas por colinearidade: {b['exog_removidas']}")

## 2.6 Disponibilidade das externas

A tabela que o enunciado pede na seção 4.1: variável por variável, se ela
estaria disponível na data real da previsão e como ela entra no modelo por
causa disso.

`conhecida` entra com o valor da própria data, porque já se sabe qual é.
`defasada` entra deslocada em `h`, porque na origem só existe o passado.
A coluna `decisao` mostra o que sobreviveu à poda por VIF.

In [ ]:
disponibilidade = (
    corr.merge(vifs.rename(columns={"exogena": "exogena"}), on=["base", "exogena"],
               how="left")
    .assign(
        disponivel_na_origem=lambda d: np.where(d["papel"] == "conhecida", "sim", "nao"),
        como_entra=lambda d: np.where(
            d["papel"] == "conhecida",
            "valor da propria data",
            "defasada em h",
        ),
    )
    [["base", "exogena", "papel", "disponivel_na_origem", "como_entra",
      "r", "p_valor", "significativa", "VIF", "decisao"]]
    .sort_values(["base", "decisao", "exogena"])
)

disponibilidade.to_csv("resultados/disponibilidade_exogenas.csv", index=False)
disponibilidade

## 2.7 Decisões consolidadas

O que a parte 2 gravou em cada base, e que a partir daqui o pipeline usa.

In [ ]:
decisoes = pd.DataFrame([{
    "base": b["nome"],
    "obs": len(b["df"]),
    "h": b["h"],
    "m": b["m"],
    "forca_sazonal": round(b["forca_sazonal"], 3),
    "m_ideal": b["m_ideal"],
    "d": b["d"],
    "D": b["D"],
    "conhecidas": ", ".join(b["exog_conhecidas"]) or "-",
    "defasadas": ", ".join(b["exog_defasadas"]) or "-",
    "removidas": ", ".join(b.get("exog_removidas", [])) or "-",
    "treino": len(b["df"]) - b["n_teste"] - b["n_validacao"],
    "validacao": b["n_validacao"],
    "teste": b["n_teste"],
} for b in bases]).set_index("base")

decisoes.to_csv("resultados/decisoes_analise.csv")
decisoes.T

# 3. Pipeline

O que é comum a todos: features, exógenas, origens e o walk-forward.
Depois, um bloco por modelo — cada um com sua função de previsão e sua
função de otimização.

## 3.1 Features

Regra da seção inteira: uma linha com data `t` só pode usar informação de até
`t - h`. Por isso todo lag é `>= h` e todo `rolling` leva `shift(h)` antes.

A mesma tabela vai para o Random Forest e para o PLS.

In [ ]:
def criar_features(base, janelas=(4, 8, 12)):
    '''Tabela de features dos modelos tabulares (Random Forest e PLS).
    
        Regra unica: uma linha com data t so pode conter informacao de ate t - h.
        Por isso todo lag e >= h e todo rolling leva shift(h) antes da janela.
    
        A mesma tabela vai para os dois modelos, para que a comparacao entre eles
        nao seja decidida por conjuntos de dados diferentes.
        '''
    h, m = base["h"], base["m"]
    y = base["df"][base["alvo"]]

    tab = pd.DataFrame(index=base["df"].index)
    tab["y"] = y

    # lags: recentes (nivel atual) e sazonais (mesmo ponto do ciclo anterior).
    # o filtro >= h nao e cosmetico: se m < h, lag_m olharia depois da origem.
    for lag in sorted({lag for lag in (h, h + 1, h + 2, h + 3, m, m + h) if lag >= h}):
        tab[f"lag_{lag}"] = y.shift(lag)

    # janelas moveis: o shift(h) antes do rolling e o que evita vazamento
    passado = y.shift(h)
    for j in janelas:
        tab[f"media_{j}"] = passado.rolling(j).mean()
        tab[f"desvio_{j}"] = passado.rolling(j).std()
    tab["delta_nivel"] = tab[f"media_{janelas[0]}"] - tab[f"media_{janelas[-1]}"]

    # calendario: vem do indice, entao e conhecido para qualquer data futura
    idx = base["df"].index
    tab["mes"] = idx.month
    tab["dia_semana"] = idx.dayofweek
    tab["semana"] = idx.isocalendar().week.astype(int).to_numpy()

    # encoding ciclico: dezembro e janeiro viram vizinhos, domingo e segunda tambem
    for col, periodo in [("mes", 12), ("dia_semana", 7), ("semana", 52)]:
        tab[f"{col}_sin"] = np.sin(2 * np.pi * tab[col] / periodo)
        tab[f"{col}_cos"] = np.cos(2 * np.pi * tab[col] / periodo)

    # externas
    for col in base["exog_conhecidas"]:
        tab[col] = base["df"][col]
    for col in base["exog_defasadas"]:
        tab[f"{col}_lag_{h}"] = base["df"][col].shift(h)

    # os lags criam NaN so no comeco da serie: essas linhas sao descartadas
    tab = tab.dropna()

    # coluna constante nao informa nada e quebra a padronizacao do PLS
    constantes = [c for c in tab.columns if c != "y" and tab[c].nunique() <= 1]
    return tab.drop(columns=constantes)

## 3.2 Exógenas do SARIMAX

A mesma `matriz_exog` da parte 2, agora sobre a série inteira — já sem as
externas que a poda por VIF derrubou.

In [ ]:
def criar_exog(base):
    '''Externas da serie inteira, ja sem as que a poda por VIF derrubou.'''
    return matriz_exog(base, base["df"])

## 3.3 Origens de previsão

O 70-30 da regra do grupo, aplicado duas vezes:

```
[------- treino -------][-- validacao --][------ teste ------]
|<----------- 70% ---------------------->|<------ 30% ------>|
```

A validação escolhe os hiperparâmetros. O teste só é tocado no final.
Os 4 modelos recebem exatamente as mesmas origens.

**O teste usa todas as origens da janela.** Já a otimização usa no máximo
`MAX_ORIGENS_VALIDACAO`, espaçadas por toda a janela de validação: cada
combinação da grade custa um walk-forward inteiro, então sem esse limite o
Pilgrim's sozinho pediria 409 origens × dezenas de combinações × 4 modelos.
O protocolo avaliado — o do teste — continua completo.

In [ ]:
def origens(base, etapa, maximo=None):
    '''Datas de origem da etapa pedida. A origem e a ultima data observada.
    
        O passo e `h`, entao os blocos previstos nao se sobrepoem e os residuos do
        teste formam uma serie continua -- o que torna a ACF deles interpretavel.
        O teste usa todas as origens; a validacao usa no maximo `maximo`, espacadas.
        '''
    idx = base["df"].index
    n, h = len(idx), base["h"]

    ini_teste = n - base["n_teste"]
    ini_val = ini_teste - base["n_validacao"]

    a, b = (ini_val, ini_teste) if etapa == "validacao" else (ini_teste, n)

    # passo = h -> blocos de previsao que nao se sobrepoem
    # a origem e a ultima data que o modelo enxerga
    lista = [idx[p] for p in range(a - 1, b - h, h)]

    # teste final: maximo=None, usa todas.
    # validacao: amostra espacada, cobrindo a janela inteira (inclui as pontas)
    if maximo and len(lista) > maximo:
        pos = np.unique(np.linspace(0, len(lista) - 1, maximo).round().astype(int))
        lista = [lista[i] for i in pos]

    return lista

def datas_futuras(base, origem):
    '''As `h` datas seguintes a origem, no calendario real da base.'''
    idx = base["df"].index
    return idx[idx.get_loc(origem) + 1:][:base["h"]]

## 3.4 Walk-forward

Anda pelas origens, chama a função de previsão do modelo em cada uma e junta
tudo num DataFrame longo. É a única parte genérica: recebe `prever` como
argumento e não sabe qual modelo está rodando.

In [ ]:
def walk_forward(prever, params, base, tab, exog, lista_origens):
    '''Roda um modelo em cada origem e devolve as previsoes fora da amostra.
    
        Unica parte generica do pipeline: recebe `prever` como argumento e nao sabe
        qual dos quatro modelos esta rodando.
        '''
    linhas = []

    for origem in lista_origens:
        datas = datas_futuras(base, origem)
        pred = prever(params, base, tab, exog, origem)

        for passo, (data, valor) in enumerate(zip(datas, pred), start=1):
            linhas.append({
                "base": base["nome"],
                "origem": origem,
                "data": data,
                "passo": passo,
                "y_real": base["df"][base["alvo"]].loc[data],
                "y_previsto": float(valor),
            })

    saida = pd.DataFrame(linhas)
    saida["residuo"] = saida["y_real"] - saida["y_previsto"]
    return saida


def mae_validacao(prever, params, base, tab, exog):
    '''MAE de uma combinacao de hiperparametros na janela de validacao.
    
        Devolve infinito quando o ajuste falha, para a combinacao simplesmente
        perder a disputa em vez de derrubar a busca inteira.
        '''
    # mesmo walk-forward, na janela de validacao e com origens amostradas
    try:
        org = origens(base, "validacao", maximo=MAX_ORIGENS_VALIDACAO)
        prev = walk_forward(prever, params, base, tab, exog, org)
        return mean_absolute_error(prev["y_real"], prev["y_previsto"])
    except Exception:
        return np.inf

## 3.5 SARIMAX

As ordens não são fixas: `d` e `D` saem dos testes de estacionariedade e o
resto é grade `(p,d,q)×(P,D,Q,m)`, em paralelo com `joblib` — mesmo
procedimento da Tarefa 03.

A decisão usa **BIC e MAE juntos**, porque os dois medem coisas diferentes:
BIC é ajuste dentro da amostra penalizado por complexidade, MAE de validação é
erro fora da amostra. BIC baixo não garante MAE baixo. Os dois são normalizados
em 0–1 (min-max) e somados; vence o menor total.

Em duas etapas, por custo: o BIC roda na grade inteira (é um ajuste por
combinação), e o MAE de validação só nas `top_bic` melhores — cada uma custa um
walk-forward completo. A normalização é feita dentro dessa lista final, que é o
conjunto entre o qual se está de fato decidindo.

A etapa do BIC fica em cache. As ordens escolhidas ficam congeladas no
walk-forward de teste.

In [ ]:
def prever_sarimax(params, base, tab, exog, origem):
    '''Ajusta na janela ate a origem e projeta `h` passos com as exogenas.'''
    y_treino = base["df"][base["alvo"]].loc[:origem]
    datas = datas_futuras(base, origem)

    ajuste = SARIMAX(
        y_treino,
        exog=None if exog is None else exog.loc[:origem],
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, maxiter=100)

    previsao = ajuste.get_forecast(
        steps=base["h"], exog=None if exog is None else exog.loc[datas]
    )
    return previsao.predicted_mean.to_numpy()

In [ ]:
def _criterios_sarimax(order, seasonal_order, y, exog):
    '''Ajusta uma combinacao e devolve (AIC, BIC), ou None se nao convergir.

    Os dois criterios penalizam complexidade de formas diferentes: o BIC pune
    mais parametros do que o AIC. Guardamos ambos porque o enunciado pede
    'AIC/BIC' e porque a discordancia entre eles e informativa no relatorio.
    '''
    try:
        ajuste = SARIMAX(
            y, exog=exog, order=order, seasonal_order=seasonal_order,
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False, maxiter=100)
        return ajuste.aic, ajuste.bic
    except Exception:
        return None


def normalizar(s):
    '''Min-max para 0-1. Se todos os valores forem iguais, devolve zeros.

    Serve para somar criterios de unidades diferentes (BIC e MAE).
    '''
    faixa = s.max() - s.min()
    return pd.Series(0.0, index=s.index) if faixa == 0 else (s - s.min()) / faixa


def otimizar_sarimax(base, tab, exog, top_bic=10, usar_cache=True):
    '''Escolhe (p,d,q)(P,D,Q,m) combinando BIC e MAE de validacao.

    Etapa 1: AIC e BIC de toda a grade, um ajuste por combinacao, em paralelo.
    Etapa 2: MAE de validacao das `top_bic` melhores, cada uma um walk-forward.
    Decisao: menor soma de BIC e MAE normalizados em 0-1 dentro dessa lista.

    Devolve (melhores_params, tabela_da_busca). A etapa 1 fica em cache.
    '''
    # corte: nada do conjunto de teste entra na escolha das ordens
    corte = base["df"].index[-base["n_teste"] - 1]
    y = base["df"][base["alvo"]].loc[:corte]
    ex = None if exog is None else exog.loc[:corte]

    # d, D e m vem da parte 2, decididos pelos testes de estacionariedade
    # e pela forca da sazonalidade
    m, d_sug, D_sug = base["m"], base["d"], base["D"]
    print(f"      d={d_sug} | D={D_sug} | m={m} (decididos na analise previa)")

    if m < 2:
        # sem sazonalidade: vira ARIMAX, so a parte nao sazonal e buscada
        combos = [((p, d, q), (0, 0, 0, 0))
                  for p in range(3) for d in range(d_sug + 1) for q in range(3)]
    else:
        combos = [
            ((p, d, q), (P, D, Q, m))
            for p in range(3) for d in range(d_sug + 1) for q in range(3)
            for P in range(3) for D in range(D_sug + 1) for Q in range(3)
        ]

    # --- etapa 1: BIC na grade inteira (um ajuste por combinacao) ---------
    cache = f"resultados/cache_sarimax_{base['nome']}.csv"
    if usar_cache and os.path.exists(cache):
        grade = pd.read_csv(cache)
        grade["params"] = grade["params"].apply(ast.literal_eval)
    else:
        print(f"      AIC/BIC de {len(combos)} combinacoes em paralelo...")
        criterios = Parallel(n_jobs=-1, backend="loky")(
            delayed(_criterios_sarimax)(order, so, y, ex) for order, so in combos
        )
        grade = pd.DataFrame([
            {"params": {"order": order, "seasonal_order": so}, "aic": ic[0], "bic": ic[1]}
            for (order, so), ic in zip(combos, criterios) if ic is not None
        ]).sort_values("bic").reset_index(drop=True)

        grade.assign(params=grade["params"].astype(str)).to_csv(cache, index=False)

    # --- etapa 2: MAE de validacao nas melhores por BIC -------------------
    busca = grade.head(top_bic).reset_index(drop=True)
    print(f"      MAE de validacao nas {len(busca)} melhores por BIC...")

    busca["mae_validacao"] = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_sarimax, p, base, tab, exog) for p in busca["params"]
    )

    # --- decisao: soma dos dois normalizados ------------------------------
    busca["bic_norm"] = normalizar(busca["bic"])
    busca["mae_norm"] = normalizar(busca["mae_validacao"])
    busca["criterio"] = "BIC+MAE normalizados"
    busca["valor"] = busca["bic_norm"] + busca["mae_norm"]

    busca = busca.sort_values("valor").reset_index(drop=True)
    return busca.loc[0, "params"], busca

## 3.6 Holt-Winters

Referência univariada: não recebe exógenas. A grade cruza tendência,
sazonalidade e amortecimento, e a escolha é pelo MAE na validação.

In [ ]:
def _ajustar_holtwinters(y_treino, params, m):
    '''Monta e ajusta o ExponentialSmoothing. Separado porque tanto a previsao
    quanto a extracao dos parametros de suavizacao precisam do mesmo ajuste.'''
    return ExponentialSmoothing(
        y_treino,
        trend=params["trend"],
        seasonal=params["seasonal"],
        damped_trend=params["damped"],
        seasonal_periods=m if params["seasonal"] else None,
        initialization_method="estimated",
    ).fit(optimized=True)


def prever_holtwinters(params, base, tab, exog, origem):
    '''Referencia univariada: so enxerga o proprio historico do alvo.'''
    y_treino = base["df"][base["alvo"]].loc[:origem]
    ajuste = _ajustar_holtwinters(y_treino, params, base["m"])
    return np.asarray(ajuste.forecast(base["h"]))


def suavizacao_holtwinters(base, params):
    '''Alpha, beta, gamma e phi estimados no fim da janela de treino.

    Eles nao sao escolhidos pela grade: o `optimized=True` os estima por maxima
    verossimilhanca. Ainda assim o enunciado pede os parametros de suavizacao
    entre os elementos minimos a investigar, entao ficam registrados aqui.

    Leitura: alpha alto = nivel reage rapido ao ultimo dado; alpha baixo = nivel
    lento e suave. Mesma logica para beta (tendencia) e gamma (sazonalidade).
    '''
    y = base["df"][base["alvo"]].iloc[:-base["n_teste"]]
    p = _ajustar_holtwinters(y, params, base["m"]).params

    return {
        "base": base["nome"],
        "trend": params["trend"],
        "seasonal": params["seasonal"],
        "damped": params["damped"],
        "m": base["m"],
        "alpha_nivel": p.get("smoothing_level"),
        "beta_tendencia": p.get("smoothing_trend"),
        "gamma_sazonal": p.get("smoothing_seasonal"),
        "phi_amortecimento": p.get("damping_trend"),
    }


def otimizar_holtwinters(base, tab, exog):
    '''Grade de tendencia x sazonalidade x amortecimento, decidida por MAE.'''
    # m=1 -> sem sazonalidade: sobra Holt com tendencia amortecida
    sazonalidades = ("add", "mul", None) if base["m"] >= 2 else (None,)

    grade = [
        {"trend": t, "seasonal": s, "damped": d}
        for t in ("add", None)
        for s in sazonalidades
        for d in (True, False)
        if not (t is None and d)          # nao existe amortecimento sem tendencia
    ]

    # Holt-Winters nao usa exogenas: passa None
    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_holtwinters, p, base, tab, None) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 3.7 Random Forest

Usa a tabela de features. `n_jobs=1` no estimador de propósito: quem paraleliza
é a grade, e aninhar os dois deixa mais lento.

A busca é **aleatória**, não exaustiva: a grade completa dos cinco
hiperparâmetros tem 48 combinações, e cada uma custa um walk-forward inteiro.
Random search com `n_amostras` sorteios cobre melhor o espaço por unidade de
tempo do que uma grade menor e exaustiva, e o `RANDOM_STATE` mantém o sorteio
reprodutível.

In [ ]:
def prever_rf(params, base, tab, exog, origem):
    '''Random Forest com estrategia direta: preve as `h` datas de uma vez.
    
        `n_jobs=1` de proposito -- quem paraleliza e a grade, e aninhar os dois
        deixa mais lento.
        '''
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1, **params)
    reg.fit(treino.drop(columns="y"), treino["y"])

    return reg.predict(futuro.drop(columns="y"))


def otimizar_rf(base, tab, exog, n_amostras=16):
    '''Random search sobre os cinco hiperparametros, decidido por MAE.
    
        A grade completa tem 48 combinacoes e cada uma custa um walk-forward
        inteiro. Busca aleatoria cobre melhor o espaco por unidade de tempo, e o
        RANDOM_STATE mantem o sorteio reprodutivel.
        '''
    completa = [
        {"n_estimators": n, "max_depth": d, "max_features": f,
         "min_samples_split": s, "min_samples_leaf": l}
        for n in (300, 600)
        for d in (None, 6, 12)
        for f in ("sqrt", 0.5)
        for s in (2, 5)
        for l in (1, 2)
    ]

    sorteio = np.random.default_rng(RANDOM_STATE)
    escolhidas = sorteio.choice(len(completa), size=min(n_amostras, len(completa)),
                                replace=False)
    grade = [completa[i] for i in sorted(escolhidas)]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_rf, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 3.8 PLS — modelo de especialização do grupo

Mesma tabela de features do Random Forest, para a comparação não ser decidida
por dados diferentes.

O `StandardScaler` não é opcional: o PLS constrói componentes maximizando
covariância com o alvo, então sem padronizar a feature de maior escala domina
tudo. `n_components` é o hiperparâmetro principal — poucos componentes
subajustam, muitos fazem o modelo voltar a ser uma regressão linear comum.

In [ ]:
def prever_pls(params, base, tab, exog, origem):
    '''PLS sobre a mesma tabela do Random Forest, com padronizacao obrigatoria.
    
        O PLS constroi componentes maximizando covariancia com o alvo: sem
        padronizar, a feature de maior escala domina todos os componentes.
        '''
    treino = tab.loc[:origem]
    futuro = tab.loc[datas_futuras(base, origem)]

    reg = make_pipeline(StandardScaler(), PLSRegression(n_components=params["n_components"]))
    reg.fit(treino.drop(columns="y"), treino["y"])

    return np.asarray(reg.predict(futuro.drop(columns="y"))).ravel()


def otimizar_pls(base, tab, exog):
    '''Varre o numero de componentes, decidido por MAE de validacao.
    
        Poucos componentes subajustam; muitos fazem o modelo convergir para uma
        regressao linear comum e perder a vantagem do PLS.
        '''
    # o limite e o numero de features disponiveis
    n_max = min(15, tab.shape[1] - 1)
    grade = [{"n_components": k} for k in range(1, n_max + 1)]

    maes = Parallel(n_jobs=-1, backend="loky")(
        delayed(mae_validacao)(prever_pls, p, base, tab, exog) for p in grade
    )

    busca = pd.DataFrame([
        {"params": p, "criterio": "MAE validacao", "valor": e} for p, e in zip(grade, maes)
    ]).sort_values("valor").reset_index(drop=True)

    return busca.loc[0, "params"], busca

## 3.9 Rodar

Para cada base e cada modelo: otimiza fora do teste → congela os
hiperparâmetros → walk-forward no teste.

In [ ]:
MODELOS = [
    ("SARIMAX",       otimizar_sarimax,      prever_sarimax),
    ("Holt-Winters",  otimizar_holtwinters,  prever_holtwinters),
    ("Random Forest", otimizar_rf,           prever_rf),
    ("PLS",           otimizar_pls,          prever_pls),
]


def rodar(bases, modelos=MODELOS):
    '''Executa as 5 bases x 4 modelos.
    
        Para cada combinacao: otimiza fora do teste, congela os hiperparametros e
        roda o walk-forward no teste. Devolve previsoes, escolhas e as buscas.
        '''
    previsoes, escolhidos, buscas = [], [], []

    for base in bases:
        tab = criar_features(base)
        exog = criar_exog(base)
        org = origens(base, "teste")

        print(f"[{base['nome']}] {len(base['df'])} obs | h={base['h']} | m={base['m']} | "
              f"{tab.shape[1] - 1} features | {len(org)} origens de teste")

        for nome, otimizar, prever in modelos:
            inicio = pd.Timestamp.now()
            print(f"   {nome}")

            params, busca = otimizar(base, tab, exog)

            # hiperparametros congelados: daqui pra frente ninguem mais mexe
            prev = walk_forward(prever, params, base, tab, exog, org)
            prev.insert(1, "modelo", nome)

            segundos = (pd.Timestamp.now() - inicio).total_seconds()
            print(f"      {params}  ({segundos:.0f}s)")

            previsoes.append(prev)
            busca["base"], busca["modelo"] = base["nome"], nome
            buscas.append(busca)
            escolhidos.append({
                "base": base["nome"],
                "modelo": nome,
                "params": str(params),
                "criterio": busca.loc[0, "criterio"],
                "valor": busca.loc[0, "valor"],
                "segundos": round(segundos, 1),
            })

    return (pd.concat(previsoes, ignore_index=True),
            pd.DataFrame(escolhidos),
            pd.concat(buscas, ignore_index=True))

In [ ]:
previsoes, parametros, buscas = rodar(bases)

previsoes.to_csv("resultados/previsoes.csv", index=False)
parametros.to_csv("resultados/hiperparametros.csv", index=False)
buscas.assign(params=buscas["params"].astype(str)).to_csv("resultados/busca.csv", index=False)

parametros

## 3.10 Parâmetros de suavização do Holt-Winters

A grade escolhe a *forma* do modelo — se tem tendência, se tem sazonalidade,
se amortece. Os parâmetros de suavização em si (`alpha`, `beta`, `gamma`,
`phi`) não são escolhidos por nós: o `optimized=True` os estima por máxima
verossimilhança a cada ajuste. Como o enunciado os lista entre os elementos
mínimos a investigar, ficam registrados aqui, estimados no fim da janela de
treino com a configuração já congelada.

Um `alpha` perto de 1 significa nível que reage quase só à última observação;
perto de 0, nível lento que quase ignora o dado novo. Mesma leitura para
`beta` na tendência e `gamma` na sazonalidade.

In [ ]:
escolhas_hw = {linha["base"]: ast.literal_eval(linha["params"])
               for _, linha in parametros.query("modelo == 'Holt-Winters'").iterrows()}

suavizacao = pd.DataFrame([
    suavizacao_holtwinters(b, escolhas_hw[b["nome"]])
    for b in bases if b["nome"] in escolhas_hw
]).set_index("base")

suavizacao.to_csv("resultados/holtwinters_suavizacao.csv")
suavizacao

# 4. Comparação

Tudo daqui pra baixo sai de `previsoes` — as previsões fora da amostra, com os
hiperparâmetros já congelados. Nada aqui é específico de base ou de modelo:
vale igual para as 5 bases e os 4 modelos.

## 4.1 MAE e ranking dentro de cada base

Bases com escalas diferentes não podem ter o MAE somado nem promediado entre
si. Por isso o MAE e o ranking são calculados **dentro** de cada base, e a
comparação entre bases é feita depois pela posição, não pelo valor.

In [ ]:
previsoes["erro_abs"] = (previsoes["y_real"] - previsoes["y_previsto"]).abs()

mae = (previsoes.groupby(["base", "modelo"])["erro_abs"]
       .mean()
       .rename("MAE")
       .reset_index())

# posicao 1 = menor MAE dentro da base
mae["posicao"] = mae.groupby("base")["MAE"].rank(method="min").astype(int)

mae.sort_values(["base", "posicao"])

In [ ]:
tabela_mae = mae.pivot(index="modelo", columns="base", values="MAE").round(3)
tabela_posicao = mae.pivot(index="modelo", columns="base", values="posicao")

print("MAE por base")
print(tabela_mae.to_string())
print()
print("Posicao por base")
print(tabela_posicao.to_string())

## 4.2 Melhor modelo em cada base

In [ ]:
melhores = (mae.loc[mae.groupby("base")["MAE"].idxmin()]
            .set_index("base")[["modelo", "MAE"]]
            .rename(columns={"modelo": "melhor_modelo", "MAE": "melhor_MAE"}))

melhores

## 4.3 Vitórias e posição média

Vitória = ter o menor MAE da base. A posição média resume o desempenho geral
do modelo sem misturar escalas — é o número que permite comparar as 5 bases.

In [ ]:
placar = (mae.groupby("modelo")
          .agg(vitorias=("posicao", lambda p: int((p == 1).sum())),
               posicao_media=("posicao", "mean"),
               melhor_posicao=("posicao", "min"),
               pior_posicao=("posicao", "max"),
               bases=("base", "nunique"))
          .sort_values(["vitorias", "posicao_media"], ascending=[False, True]))

placar["posicao_media"] = placar["posicao_media"].round(2)
placar

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].barh(placar.index, placar["vitorias"], color="steelblue")
axes[0].set_xlabel("vitorias (menor MAE na base)")
axes[0].set_title("Vitorias por modelo", loc="left")

axes[1].barh(placar.index, placar["posicao_media"], color="darkorange")
axes[1].set_xlabel("posicao media (menor e melhor)")
axes[1].set_title("Posicao media", loc="left")

for ax in axes:
    ax.invert_yaxis()
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

plt.tight_layout()
plt.show()

## 4.4 Resíduos

A tabela de Ljung-Box sai para as 20 combinações — é ela que vai no corpo do
relatório. Depois vêm os gráficos: primeiro o **melhor modelo de cada base**,
que é o que entra no corpo do texto, e em seguida as **20 combinações**, que
vão para o apêndice. O enunciado pede métodos estatísticos e gráficos para
cada combinação entre base e modelo.

Como o walk-forward usa passo `= h`, os blocos de previsão não se sobrepõem e
os resíduos do teste formam uma série contínua no tempo — é isso que torna a
ACF interpretável aqui.

In [ ]:
def ljung_box(residuos, lags=10):
    '''Estatistica e p-valor do teste de Ljung-Box nos residuos.
    
        p > 0.05: nao se rejeita a hipotese de ausencia de autocorrelacao, ou seja,
        os residuos passam por ruido branco. p baixo: sobrou padrao no erro.
        '''
    lags = max(1, min(lags, len(residuos) // 5))
    r = acorr_ljungbox(residuos, lags=[lags], return_df=True)
    return lags, float(r["lb_stat"].iloc[0]), float(r["lb_pvalue"].iloc[0])


linhas = []
for (b, mdl), g in previsoes.groupby(["base", "modelo"]):
    residuos = g.sort_values("data")["residuo"]
    lags, stat, p = ljung_box(residuos)
    linhas.append({
        "base": b,
        "modelo": mdl,
        "n": len(residuos),
        "lags": lags,
        "vies": residuos.mean(),          # media do residuo: > 0 subestima, < 0 superestima
        "desvio": residuos.std(),
        "lb_stat": stat,
        "p_valor": p,
        "ruido_branco": "sim" if p > 0.05 else "nao",
    })

tabela_ljung = pd.DataFrame(linhas).sort_values(["base", "modelo"]).round(4)
tabela_ljung.to_csv("resultados/ljung_box.csv", index=False)

tabela_ljung

In [ ]:
def analisar_residuos(base_nome, modelo_nome):
    '''Residuos no tempo, ACF e distribuicao de uma combinacao base x modelo.'''
    g = (previsoes.query("base == @base_nome and modelo == @modelo_nome")
         .sort_values("data"))
    residuos = g.set_index("data")["residuo"]
    lags, stat, p = ljung_box(residuos)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

    # residuos no tempo: procura tendencia, viés e mudanca de variancia
    axes[0].plot(residuos.index, residuos.values, color="steelblue",
                 marker="o", markersize=3, linewidth=1)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].axhline(residuos.mean(), color="tomato", linestyle="--", linewidth=1,
                    label=f"media = {residuos.mean():.2f}")
    axes[0].set_title("Residuos no tempo", loc="left")
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis="x", rotation=45)

    # ACF: padrao remanescente que o modelo nao capturou
    plot_acf(residuos, lags=max(1, min(20, len(residuos) // 2 - 1)), ax=axes[1])
    axes[1].set_title("ACF dos residuos", loc="left")

    axes[2].hist(residuos.values, bins=15, color="steelblue", edgecolor="white")
    axes[2].axvline(0, color="black", linewidth=0.8)
    axes[2].set_title("Distribuicao", loc="left")

    fig.suptitle(f"{base_nome} - {modelo_nome}", y=1.04, fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"Ljung-Box ({lags} lags): estatistica = {stat:.3f} | p-valor = {p:.4f}")
    print("  -> residuos se comportam como ruido branco" if p > 0.05
          else "  -> ainda ha autocorrelacao: sobrou padrao no erro")
    print(f"Vies (media do residuo) = {residuos.mean():.3f} | desvio = {residuos.std():.3f}")

In [ ]:
# corpo do relatorio: o melhor modelo de cada base
for base_nome, linha in melhores.iterrows():
    analisar_residuos(base_nome, linha["melhor_modelo"])

### Apêndice — as 20 combinações

Mesma análise para tudo. O melhor modelo de cada base vem marcado, para não se
perder a referência ao olhar a sequência inteira.

In [ ]:
for base_nome in previsoes["base"].unique():
    melhor = melhores.loc[base_nome, "melhor_modelo"]

    for modelo_nome in previsoes.query("base == @base_nome")["modelo"].unique():
        marca = "  [melhor da base]" if modelo_nome == melhor else ""
        print(f"=== {base_nome} - {modelo_nome}{marca} ===")
        analisar_residuos(base_nome, modelo_nome)

## 4.5 Importância das features

A fazer: importância nativa e permutation importance no Random Forest;
coeficientes padronizados e VIP scores no PLS. Destacar onde as externas
aparecem e retomar a disponibilidade delas.